<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 Krea-2-Turbo - Fast Text-to-Image Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>12B DiT · 8-Step Distilled · INT8 Tensor-Core Kernels | Powered by Wan2GP + mmgp</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is this notebook?

**Fast Text-to-Image Generation** using the **Krea-2-Turbo** model (12B parameters) powered by the **Wan2GP** engine and **mmgp** offloading.

- **INT8 Tensor Cores:** the int8 weights run on real INT8 matmul kernels (Comfy Kitchen) instead of being dequantized to FP16 every step
- **FP16 on T4:** the transformer and VAE run in FP16 (the T4 has no BF16 tensor cores)
- **Both T4s used:** the transformer owns GPU 0 (pinned + async offloading, never swapped out); the text encoder and VAE run on GPU 1
- **No wasted tokens:** prompt padding is stripped, so every step only processes the real text + image tokens
- **8 steps** (distilled turbo model, no CFG) · negative prompts via **NAG** · prompt enhancer & style presets

### Quick Start
1. **Settings → Accelerator → GPU T4 x2**
2. **Turn on Internet** in the Settings sidebar
3. **Run All** - setup + downloads take a few minutes the first time
4. Open the **Gradio link** or the **Cloudflare link** (`*.trycloudflare.com`) printed by the last cell
5. Generated images are also saved to `/kaggle/working/outputs`

---
## Step 1: Setup - Wan2GP (pinned) & Minimal Dependencies

Wan2GP is pinned to a tested commit so upstream changes can't break the notebook. Only the packages the Krea-2 pipeline actually imports are installed (Wan2GP's full `requirements.txt` pulls in 80+ audio/video/vision packages and a nightly `onnxruntime` build that makes the whole install fail).

In [ ]:
import os, sys, subprocess, psutil

print('=== Kaggle GPU T4 x2 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')
try:
    subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'], check=True)
except Exception:
    print('❌ No GPU. Go to Settings → Accelerator → GPU T4 x2')

# ---- Wan2GP, pinned to a tested commit ----
WAN2GP_COMMIT = '2345ae148f82740f66e82c41292dbbdd592e713d'  # 2026-09-25
REPO_DIR = 'Wan2GP'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'init', '-q', REPO_DIR], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'add', 'origin', 'https://github.com/DeepBeepMeep/Wan2GP.git'], check=True)
print('\n📥 Fetching Wan2GP...')
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '-q', '--depth', '1', 'origin', WAN2GP_COMMIT], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', '--force', 'FETCH_HEAD'], check=True)  # also drops patches from earlier runs

# ---- Only what the Krea-2 pipeline imports (Kaggle's PyTorch/torchvision are kept) ----
print('📦 Installing dependencies...')
PACKAGES = [
    'mmgp==3.8.1',            # offloading engine used by Wan2GP
    'comfy-kitchen==0.2.35',  # INT8 tensor-core kernels (T4 supported)
    'transformers==4.54.0',   # version Wan2GP's Qwen3-VL text encoder is written against
    'diffusers==0.36.0',      # Qwen image VAE base classes
    'gradio==5.50.0',
    'optimum-quanto', 'accelerate', 'einops', 'hf_xet',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts',
                '--timeout', '120', '--retries', '5', *PACKAGES], check=True)

# ---- Let the notebook choose the dtype (Wan2GP hardcodes BF16, which the T4 can only emulate) ----
main_py = os.path.join(REPO_DIR, 'models', 'krea2', 'krea2_main.py')
src = open(main_py).read()
PATCH_FROM = '        dtype = torch.bfloat16\n'
if src.count(PATCH_FROM) != 1:
    raise RuntimeError(f'❌ Unexpected {main_py} contents - dtype patch not applied')
with open(main_py, 'w') as f:
    f.write(src.replace(PATCH_FROM, '        # dtype = torch.bfloat16  # AIQUEST: FP16 on T4 for tensor-core speed\n'))

print('✅ Wan2GP ready, dependencies installed and patched!')

---
## Step 2: Download INT8 Models

Downloads the INT8 quantized Krea-2-Turbo transformer (~13.5 GB), the Qwen3-VL-4B text encoder (~4.4 GB, incl. its tokenizer/processor configs) and the Qwen image VAE into `/kaggle/tmp` (outside the 20 GB `/kaggle/working` limit), symlinked into `Wan2GP/ckpts`. Files are fetched in parallel.

In [ ]:
import os, shutil
from concurrent.futures import ThreadPoolExecutor
os.environ['HF_HOME'] = '/kaggle/tmp/hf_home'
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'  # no duplicate chunk cache on disk
from huggingface_hub import hf_hub_download

KREA_REPO = 'DeepBeepMeep/krea-2'
QWEN_IMAGE_REPO = 'DeepBeepMeep/Qwen_image'
CKPT_DIR = 'Wan2GP/ckpts'
TMP_DIR = '/kaggle/tmp/models'
TE = 'Qwen3-VL-4B-Instruct'

FILES = [
    (KREA_REPO, 'Krea2Turbo_quanto_bf16_int8.safetensors'),
    (KREA_REPO, f'{TE}/{TE}_quanto_bf16_int8.safetensors'),
    *[(KREA_REPO, f'{TE}/{f}') for f in ['config.json', 'tokenizer.json', 'tokenizer_config.json',
                                          'chat_template.jinja', 'preprocessor_config.json']],
    (QWEN_IMAGE_REPO, 'qwen_vae.safetensors'),
    (QWEN_IMAGE_REPO, 'qwen_vae_config.json'),
]

def fetch(repo, filename):
    dest = os.path.join(CKPT_DIR, filename)
    if os.path.exists(dest):
        return f'  ✓ Already present: {filename}'
    path = hf_hub_download(repo_id=repo, filename=filename, local_dir=TMP_DIR)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.lexists(dest):
        os.remove(dest)  # stale symlink from an earlier session
    os.symlink(os.path.abspath(path), dest)
    return f'  ✓ {filename}'

print('📥 Downloading models...')
with ThreadPoolExecutor(max_workers=4) as pool:
    for line in pool.map(lambda item: fetch(*item), FILES):
        print(line)

shutil.rmtree('/kaggle/tmp/hf_home', ignore_errors=True)
os.system('df -h /kaggle/working /kaggle/tmp')
print('\n✅ All downloads complete!')

---
## Step 3: Write the Generation Script

Loads the model once (INT8 kernels, FP16, pinned offloading), runs a short warm-up so the first image is fast, and builds the branded Gradio UI.

In [ ]:
%%writefile run_krea_turbo.py
import gc
import os
import re
import sys
import time
import random
import traceback
import warnings
import zipfile

warnings.filterwarnings("ignore", message=".*parameter in the Blocks constructor.*")
warnings.filterwarnings("ignore", message=".*Mismatch dtype between input and weight.*")

# Keep transformers/diffusers from importing Kaggle's TensorFlow/JAX: faster startup, no protobuf/XLA spam
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_JAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
CKPT_DIR = "ckpts"
# Images are saved in the working folder (/kaggle/working/outputs), visible in Kaggle's Output panel
OUTPUT_DIR = os.path.join(os.path.dirname(WAN2GP_DIR), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

# Krea2's VAE only needs 3 helpers from models/wan/modules/vae.py. Register the Wan packages without
# running their __init__, which imports the whole Wan video stack (SCAIL/smplfitter, onnxruntime, rembg...)
import importlib.machinery, types
for _pkg in ("models.wan", "models.wan.modules"):
    _mod = types.ModuleType(_pkg)
    _mod.__path__ = [os.path.join(WAN2GP_DIR, *_pkg.split("."))]
    _mod.__spec__ = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod.__spec__.submodule_search_locations = _mod.__path__
    sys.modules[_pkg] = _mod

import psutil
import torch
from PIL import Image
import gradio as gr
from mmgp import offload
from optimum.quanto.tensor.weights import qbytes
from shared.utils import files_locator as fl
from shared.kernels import int8_backend, quanto_int8_inject
from models.krea2 import krea2_mmdit
from models.krea2.krea2_handler import family_handler

fl.set_checkpoints_paths([CKPT_DIR, "."])

# Text encoder + VAE go on the second T4 so the transformer never leaves GPU 0 (no ~15 GB of
# weight swapping per image). Set to False to run everything on one GPU.
USE_SECOND_GPU = True

# ==== GPU INFO ====
gpu = torch.cuda.get_device_properties(0)
ram = psutil.virtual_memory()
MAIN = torch.device("cuda:0")
AUX = torch.device("cuda:1") if USE_SECOND_GPU and torch.cuda.device_count() > 1 else None
# T4 (sm75) has no BF16 tensor cores: run everything in FP16 there, keep native BF16 on Ampere+
DTYPE = torch.bfloat16 if gpu.major >= 8 else torch.float16
print("=" * 65)
print("🎨 Krea-2-Turbo - Fast Text-to-Image Generator | AIQUEST Academy")
print(f"⚡ GPU: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GB VRAM, sm_{gpu.major}{gpu.minor}) x{torch.cuda.device_count()} | "
      f"RAM: {ram.total / 1024**3:.1f} GB | dtype: {str(DTYPE).replace('torch.', '')}")
print(f"🧩 Layout: transformer → cuda:0 | text encoder + VAE → {AUX or 'cuda:0 (shared)'}")
print("=" * 65)
sys.stdout.flush()

# ==== INT8 Tensor Core kernels (Wan2GP's "INT8 Kernels: Auto") ====
# Both weight files are quanto int8: without this every int8 layer is dequantized and run as an
# FP16 matmul. Auto picks Comfy Kitchen, then Triton, then falls back to PyTorch.
try:
    int8_backend.configure("auto", 1)
except Exception as e:
    print(f"⚠️ INT8 kernels unavailable, using PyTorch matmul: {e}")
INT8_LABELS = {"kitchen": "Comfy Kitchen INT8", "triton": "Triton INT8", "pytorch": "PyTorch"}
# Wan2GP splits every INT8 matmul into 16 MB row tiles (~2000 kernel calls per step); 64 MB tiles
# cut that ~4x at a small VRAM cost
int8_backend._SCRATCH_BYTES = 64 * 1024 * 1024

# ==== LOAD INT8 QUANTIZED MODEL VIA WAN2GP ====
print("\nLoading Krea-2-Turbo model (quanto int8)...")
sys.stdout.flush()

BASE_MODEL_TYPE = "krea2_turbo"
model_def = family_handler.query_model_def(BASE_MODEL_TYPE, {})
transformer_path = os.path.join(CKPT_DIR, "Krea2Turbo_quanto_bf16_int8.safetensors")
text_encoder_path = os.path.join(CKPT_DIR, "Qwen3-VL-4B-Instruct", "Qwen3-VL-4B-Instruct_quanto_bf16_int8.safetensors")
for path in (transformer_path, text_encoder_path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ {path} not found - run the download cell first.")

krea_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type=BASE_MODEL_TYPE,
    base_model_type=BASE_MODEL_TYPE,
    model_def=model_def,
    dtype=DTYPE,
    VAE_dtype=DTYPE,
    text_encoder_filename=text_encoder_path,
)
# Wan2GP keeps the VAE in its file dtype (BF16), which a T4 can only emulate
krea_model.vae.to(device=AUX or "cpu", dtype=DTYPE)
if AUX is not None:
    krea_model.text_encoder.language_model.to(AUX)
    pipe = {"transformer": pipe["transformer"]}

# ==== Apply mmgp profile ====
# Transformer (~12.6 GB) fully pinned: all weight transfers are async DMA. It keeps as many blocks
# resident as 80% of VRAM allows; the rest stream in behind compute.
print("\nApplying mmgp profile (pinned + async transfers)...")
sys.stdout.flush()
offload.profile(
    pipe,
    profile_no=2,
    quantizeTransformer=False,
    convertWeightsFloatTo=DTYPE,
    pinnedMemory=list(pipe) if AUX is None else ["transformer"],
    perc_reserved_mem_max=0.6,
    asyncTransfers=True,
    budgets={"transformer": 13000, "text_encoder": 5000, "*": 3000},
)
offload.shared_state["_attention"] = "sdpa"  # memory-efficient SDPA kernel (FlashAttention needs sm80+)

# ==== No padding anywhere (exact: padded tokens are always masked out) ====
# 1) Prompts were padded to 512 tokens: the text encoder now only sees the real tokens (padding sat
#    between the prompt and the chat suffix, masked, with positions from cumsum(mask)).
class _NoPadTokenizer:
    def __init__(self, tokenizer):
        self._tokenizer = tokenizer
    def __call__(self, *args, **kwargs):
        if kwargs.get("padding") == "max_length":
            kwargs["padding"] = "longest"
        return self._tokenizer(*args, **kwargs)
    def __getattr__(self, name):
        return getattr(self._tokenizer, name)

krea_model.pipeline.encoder.tokenizer = _NoPadTokenizer(krea_model.pipeline.encoder.tokenizer)

# 2) The DiT padded the text+image sequence to a multiple of 256 with masked rows. Text tokens have no
#    positional encoding, so dropping the padding is exact; with nothing left to mask, attention runs
#    unmasked (NAG keeps its mask: the negative prompt is padded to the positive prompt's length).
def _build_stream_nopad(self, img, context, pos, mask, freqs=None):
    txtlen, imglen = context.shape[1], img.shape[1]
    combined = img.new_empty(img.shape[0], txtlen + imglen, img.shape[-1])
    combined[:, :txtlen].copy_(context)
    combined[:, txtlen:].copy_(img)
    del context
    if freqs is None:
        freqs = self.posemb(pos).to(combined.dtype)
    if not getattr(self, "_aiq_keep_mask", False) and bool(mask.all()):
        return combined, txtlen, imglen, freqs, None
    return combined, txtlen, imglen, freqs, krea2_mmdit.key_padding_mask(mask)

krea2_mmdit.SingleStreamDiT._build_stream = _build_stream_nopad

# ==== Text encoder: exact INT8 path, on the second GPU ====
# LLM activation outliers do not survive INT8 activation quantization; the encoder runs once per
# new prompt (results are cached), so this costs nothing per step.
_QBYTES_LINEAR = qbytes.WeightQBytesLinearFunction
_encode_prompts_fast = krea_model.pipeline._encode_prompts
PHASE_TIMES = {}

def _encode_prompts_exact(prompts, device, dtype, images=None):
    t0 = time.time()
    fast_forward = _QBYTES_LINEAR.__dict__["forward"]
    _QBYTES_LINEAR.forward = staticmethod(quanto_int8_inject._default_quanto_qbytes_linear_forward)
    try:
        if AUX is None:
            return _encode_prompts_fast(prompts, device, dtype, images=images)
        with torch.cuda.device(AUX):
            hiddens, masks = _encode_prompts_fast(prompts, AUX, dtype, images=images)
        if hiddens is None:
            return None, None
        return hiddens.to(device), masks.to(device)
    finally:
        _QBYTES_LINEAR.forward = fast_forward
        PHASE_TIMES["text"] = PHASE_TIMES.get("text", 0.0) + time.time() - t0

krea_model.pipeline._encode_prompts = _encode_prompts_exact

# ==== VAE decode on the second GPU ====
_decode_fast = krea_model.pipeline._decode_latents_to_cpu_uint8

def _decode_timed(latents):
    t0 = time.time()
    try:
        if AUX is None:
            return _decode_fast(latents)
        with torch.cuda.device(AUX):
            return _decode_fast(latents.to(AUX))
    finally:
        PHASE_TIMES["vae"] = time.time() - t0

krea_model.pipeline._decode_latents_to_cpu_uint8 = _decode_timed


def run_krea(**kwargs):
    """Generate one image; if the INT8 kernels ever produce NaN/Inf, retry on the exact path."""
    PHASE_TIMES.clear()
    krea_model.transformer._aiq_keep_mask = float(kwargs.get("NAG_scale", 1.0)) > 1.0
    try:
        return krea_model.generate(**kwargs)
    except RuntimeError as e:
        if "non-finite" not in str(e) or int8_backend._backend == "pytorch":
            raise
        print("⚠️ INT8 kernels produced non-finite values - switching to the exact INT8 path and retrying.")
        int8_backend.configure("disabled", 1)
        return krea_model.generate(**kwargs)


# ==== Warm-up: loads weights onto the GPU, initialises CUDA kernels and validates the INT8 path ====
print("\nWarming up (1 step @ 512x512)...")
sys.stdout.flush()
_t0 = time.time()
with torch.inference_mode():
    run_krea(seed=0, input_prompt="a photo of a red apple on a wooden table", n_prompt=None, sampling_steps=1,
             width=512, height=512, guide_scale=0.0, batch_size=1, loras_slists={"phase1": []})
gc.collect(); torch.cuda.empty_cache()
print(f"✅ Model ready in {time.time() - _t0:.1f}s warm-up | INT8 kernels: {INT8_LABELS[int8_backend._backend]} | Attention: SDPA")
sys.stdout.flush()

# ==== PROMPT ENHANCER LOGIC ====
STYLE_MODIFIERS = {
    "Cinematic": "cinematic lighting, dramatic shadows, 8k resolution, highly detailed, film grain, masterpiece",
    "Photographic": "professional photography, 35mm lens, f/2.8, depth of field, natural lighting, highly detailed, photorealistic",
    "Anime": "anime key visual, vibrant color palette, clean line art, highly detailed digital illustration, masterpiece",
    "Cyberpunk": "cyberpunk aesthetic, glowing neon lights, futuristic city streets, volumetric smoke, high contrast, ray tracing",
    "Fantasy": "mythical fantasy landscape, glowing magical particles, ethereal light, whimsical details, digital painting, masterpiece",
}

def expand_and_enhance_prompt(prompt):
    if not prompt:
        return ""
    prompt_lower = prompt.lower()

    # Avoid double enhancement if already highly detailed
    if any(k in prompt_lower for k in ["photorealistic", "hyperrealistic", "highly detailed", "8k resolution"]):
        return prompt

    # Categories
    is_portrait = any(w in prompt_lower for w in ["person", "woman", "man", "girl", "boy", "portrait", "face", "model", "lady", "guy"])
    is_sci_fi = any(w in prompt_lower for w in ["cyberpunk", "futuristic", "sci-fi", "robot", "spaceship", "alien", "neon", "technology"])
    is_landscape = any(w in prompt_lower for w in ["forest", "mountain", "ocean", "nature", "landscape", "lake", "river", "sky", "view", "sunset", "sunrise"])
    is_fantasy = any(w in prompt_lower for w in ["dragon", "magic", "wizard", "elf", "fairy", "castle", "mythical", "ancient", "fantasy"])
    is_anime_art = any(w in prompt_lower for w in ["anime", "illustration", "drawing", "painting", "art", "sketch", "digital art", "vector"])

    if is_portrait:
        modifiers = [
            "captured on 85mm lens, f/1.8, natural skin texture, realistic catchlights in eyes, professional studio lighting, depth of field, stunning portrait, highly detailed, 8k",
            "candid photography, warm natural light, soft shadows, detailed facial features, cinematic color grading, photorealistic, masterpiece",
            "dramatic side-lighting, highly detailed skin pores, close-up shot, portra 400 film style, high resolution, sharp focus, exquisite composition"
        ]
    elif is_sci_fi:
        modifiers = [
            "cyberpunk aesthetic, neon glow, wet pavement with reflections, volumetric fog, high-tech details, dark moody atmosphere, cinematic lighting, ray tracing, 8k resolution",
            "futuristic design, industrial sci-fi details, cinematic composition, metallic surfaces, sharp contrast, volumetric light rays, octane render style, highly detailed",
            "dystopian atmosphere, high-tech elements, dramatic shadows, glowing circuits, ultra-detailed, cinematic wide shot, photorealistic, masterfully crafted"
        ]
    elif is_landscape:
        modifiers = [
            "golden hour sunset lighting, national geographic style, epic scale, volumetric mist, wide-angle lens, breathtaking scenery, sharp focus, highly detailed, 8k",
            "dramatic skies, natural lighting, crisp details, long exposure water reflections, gorgeous composition, masterpiece, photorealistic, ultra-detailed landscape",
            "atmospheric haze, majestic mountain peaks, rays of sun piercing through clouds, highly detailed textures, vibrant colors, stunning nature photography"
        ]
    elif is_fantasy:
        modifiers = [
            "mythical atmosphere, glowing magical particles, ethereal light, highly detailed fantasy illustration, masterpiece, epic scale, whimsical, digital painting, 8k",
            "dark fantasy style, dramatic moody lighting, ancient runic details, volumetric rays, gorgeous composition, hyper-detailed, stunning fantasy art",
            "enchanted forest lighting, magical glow, intricate details, breathtaking fantasy landscape, majestic, high-resolution digital masterpiece"
        ]
    elif is_anime_art:
        modifiers = [
            "stunning digital painting, vibrant color palette, highly detailed illustration, clean line art, beautiful lighting, anime key visual style, masterpiece",
            "concept art illustration, detailed background, soft lighting, dramatic composition, masterfully painted, artist station style, gorgeous visual",
            "vector illustration, clean details, smooth gradients, modern digital art style, highly detailed, creative graphic design"
        ]
    else:
        modifiers = [
            "highly detailed, photorealistic, cinematic lighting, masterpiece, stunning visual, sharp focus, 8k resolution, intricate textures",
            "professional photography, depth of field, dramatic lighting, rich colors, soft shadows, sharp focus, highly detailed, 8k",
            "award-winning photography, volumetric lighting, hyper-realistic, photorealistic, exquisite details, gorgeous composition, warm glow"
        ]
    return f"{prompt}, {random.choice(modifiers)}"

def apply_style(prompt, style_preset):
    modifier = STYLE_MODIFIERS.get(style_preset)
    if not modifier or modifier in prompt:
        return prompt
    return f"{prompt}, {modifier}" if prompt else modifier

def manual_enhance_trigger(prompt, style_preset):
    return apply_style(expand_and_enhance_prompt(prompt or ""), style_preset)

# ==== GENERATION FUNCTION ====
ASPECT_RATIOS = {"16:9 Landscape": (16, 9), "9:16 Portrait": (9, 16), "1:1 Square": (1, 1), "4:3 Standard": (4, 3), "3:4 Portrait": (3, 4)}
RESOLUTIONS = {"1024px (Standard)": 1024, "1536px (High)": 1536, "2048px (2K Ultra)": 2048}

def resolve_size(aspect_ratio, resolution):
    # Longest side = base resolution, both sides divisible by 16 (Krea2 latent patch alignment)
    base = RESOLUTIONS.get(resolution, 1024)
    rw, rh = ASPECT_RATIOS.get(aspect_ratio, (1, 1))
    if rw >= rh:
        return base, max(16, int(base * rh / rw) // 16 * 16)
    return max(16, int(base * rw / rh) // 16 * 16), base

def stop_generation():
    krea_model._interrupt = True
    return "⏹️ Stopping after the current step..."

@torch.inference_mode()
def generate_image(prompt, negative_prompt, style_preset, steps, aspect_ratio, resolution, seed, num_images, nag_scale, progress=gr.Progress()):
    try:
        krea_model._interrupt = False
        prompt = apply_style((prompt or "").strip(), style_preset)
        if not prompt:
            yield gr.update(), "❌ Please enter a prompt.", gr.update(visible=False)
            return
        negative_prompt = (negative_prompt or "").strip()
        steps, num_images = int(steps), int(num_images)
        width, height = resolve_size(aspect_ratio, resolution)
        initial_seed = random.randint(0, 2**32 - 1) if seed is None or seed < 0 else int(seed)
        # Turbo runs without CFG; a negative prompt is applied through NAG (Normalized Attention Guidance)
        nag = float(nag_scale) if negative_prompt else 1.0

        yield gr.update(value=None, selected_index=None), "⏳ Initializing generation...", gr.update(visible=False)

        paths, seeds_used, start_time = [], [], time.time()
        for idx in range(num_images):
            current_seed = (initial_seed + idx) % (2**32)

            def cb(step_idx, latent=None, is_start=False, override_num_inference_steps=None, progress_title=None, **kwargs):
                if progress_title:  # text encoder layers / VAE tiles
                    total = override_num_inference_steps or 1
                    progress(min(idx / num_images, 0.99), desc=f"Image {idx+1}/{num_images} - {progress_title} {max(step_idx + 1, 0)}/{total}")
                elif step_idx >= 0:
                    progress(min((idx + (step_idx + 1) / steps) / num_images, 0.99), desc=f"Image {idx+1}/{num_images} - Step {step_idx+1}/{steps}")

            print(f"Generating image {idx+1}/{num_images}: {width}x{height}, steps={steps}, seed={current_seed}")
            sys.stdout.flush()
            t_img = time.time()
            result = run_krea(
                seed=current_seed,
                input_prompt=prompt,
                n_prompt=negative_prompt or None,
                sampling_steps=steps,
                width=width,
                height=height,
                guide_scale=0.0,
                NAG_scale=nag,
                batch_size=1,
                callback=cb,
                loras_slists={"phase1": []},
            )
            if result is None:
                if krea_model._interrupt:
                    break
                raise RuntimeError(f"Generation of image {idx+1} failed (returned None).")

            # Output tensor [3, 1, H, W] uint8 on CPU -> PNG saved straight to the output folder
            img_path = os.path.join(OUTPUT_DIR, f"krea_{time.strftime('%Y%m%d_%H%M%S')}_seed{current_seed}.png")
            Image.fromarray(result[:, 0].permute(1, 2, 0).contiguous().numpy()).save(img_path)
            paths.append(img_path)
            seeds_used.append(current_seed)
            total = time.time() - t_img
            text_t, vae_t = PHASE_TIMES.get("text", 0.0), PHASE_TIMES.get("vae", 0.0)
            print(f"  ✅ {total:.1f}s (text {text_t:.1f}s · denoise {total - text_t - vae_t:.1f}s · VAE {vae_t:.1f}s) -> {img_path}")
            yield gr.update(value=paths, selected_index=0), f"⏳ Generated {len(paths)}/{num_images} images...", gr.update(visible=False)

        if not paths:
            yield gr.update(value=None, selected_index=None), "⏹️ Generation stopped.", gr.update(visible=False)
            return

        zip_path = None
        if len(paths) > 1:
            zip_path = os.path.join(OUTPUT_DIR, f"krea_batch_{time.strftime('%Y%m%d_%H%M%S')}.zip")
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zipf:  # PNGs are already compressed
                for path in paths:
                    zipf.write(path, os.path.basename(path))

        elapsed = time.time() - start_time
        stopped = " (stopped early)" if krea_model._interrupt else ""
        status = (f"✅ Generated {len(paths)} image(s) in {elapsed:.1f}s ({elapsed / len(paths):.1f}s/image){stopped} | "
                  f"Seeds: {seeds_used} | {width}x{height} | {steps} steps | INT8: {INT8_LABELS[int8_backend._backend]}")
        yield gr.update(value=paths, selected_index=0), status, gr.update(value=zip_path, visible=zip_path is not None)
    except Exception as e:
        traceback.print_exc()
        yield gr.update(value=None, selected_index=None), f"❌ Error: {str(e)}", gr.update(visible=False)
    finally:
        krea_model._interrupt = False
        gc.collect()


# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; font-size: 13px; }
.footer a { color: #764ba2; text-decoration: none; margin: 0 8px; font-weight: 600; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">⚡ Krea-2-Turbo - Fast Text-to-Image Generator</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; 12B DiT · 8-Step Distilled · INT8 Tensor Cores</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

STATUS_HTML = (f'<div class="status-line">🖥️ {gpu.name}{" x2 (DiT | text+VAE)" if AUX is not None else ""} · {str(DTYPE).replace("torch.", "").upper()} · '
               f'INT8 kernels: <strong>{INT8_LABELS[int8_backend._backend]}</strong> · Attention: SDPA</div>')

FOOTER_HTML = """
<div class="footer">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · Krea-2-Turbo 12B · Wan2GP INT8 Edition<br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank">YouTube</a> |
  <a href="https://x.com/aiquestacademy" target="_blank">X (Twitter)</a> |
  <a href="https://aiquest.site" target="_blank">aiquest.site</a>
</div>
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="Krea-2-Turbo Image Generator | AIQUEST") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Row(equal_height=True):
                prompt = gr.Textbox(
                    label="📝 Prompt",
                    lines=3,
                    placeholder="A small, dark-colored cat is captured mid-stride, walking down the center of a narrow, abandoned street.",
                    scale=4
                )
                enhance_btn = gr.Button("✨ Enhance", variant="secondary", scale=1)

            with gr.Row():
                style_preset = gr.Dropdown(
                    label="🎨 Style Preset",
                    choices=["None"] + list(STYLE_MODIFIERS),
                    value="None"
                )
                resolution = gr.Dropdown(
                    label="🖥️ Base Resolution",
                    choices=list(RESOLUTIONS),
                    value="1024px (Standard)",
                    info="Longest side. 1024px is fastest; 2K is much slower on a T4."
                )

            aspect_ratio = gr.Dropdown(
                label="📏 Aspect Ratio",
                choices=list(ASPECT_RATIOS),
                value="16:9 Landscape"
            )

            with gr.Row():
                steps = gr.Slider(
                    label="⚡ Diffusion Steps",
                    minimum=1, maximum=12, step=1, value=8,
                    info="8 steps is recommended (distilled model)."
                )
                num_images = gr.Slider(
                    label="🖼️ Number of Images",
                    minimum=1, maximum=4, step=1, value=1,
                    info="Generated one after another (seed +1 each)."
                )

            with gr.Accordion("⚙️ Advanced Settings", open=False):
                negative_prompt = gr.Textbox(
                    label="🚫 Negative Prompt (optional)",
                    placeholder="low quality, blurry, distorted, bad proportions, bad anatomy",
                    lines=2
                )
                nag_scale = gr.Slider(
                    label="🧲 NAG Strength",
                    minimum=1.0, maximum=1.5, step=0.01, value=1.25,
                    info="How strongly the negative prompt is applied. Only used when a negative prompt is set (adds some time per step)."
                )
                seed = gr.Number(
                    label="🎲 Seed (-1 for Random)",
                    value=-1,
                    precision=0
                )

            with gr.Row():
                gen_btn = gr.Button("🎨 Generate", variant="primary", size="lg")
                stop_btn = gr.Button("⏹️ Stop", variant="stop", size="lg")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg")

        with gr.Column(scale=1):
            image_out = gr.Gallery(
                label="🖼️ Generated Images",
                columns=2,
                rows=2,
                object_fit="contain",
                preview=True
            )
            zip_out = gr.File(label="📦 Download All Images (ZIP)", visible=False)
            status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

    enhance_btn.click(
        fn=manual_enhance_trigger,
        inputs=[prompt, style_preset],
        outputs=[prompt]
    )

    gen_btn.click(
        fn=generate_image,
        inputs=[prompt, negative_prompt, style_preset, steps, aspect_ratio, resolution, seed, num_images, nag_scale],
        outputs=[image_out, status_out, zip_out],
        concurrency_limit=1,
    )

    stop_btn.click(fn=stop_generation, outputs=[status_out], queue=False)

    clear_btn.click(
        fn=lambda: ("", "", 8, "16:9 Landscape", "1024px (Standard)", -1, 1, "None", 1.25, gr.update(value=None, selected_index=None), "", gr.update(visible=False)),
        inputs=[],
        outputs=[prompt, negative_prompt, steps, aspect_ratio, resolution, seed, num_images, style_preset, nag_scale, image_out, status_out, zip_out]
    )

    gr.HTML(FOOTER_HTML)

# ==== CLOUDFLARE QUICK TUNNEL (second public link in case the Gradio share link fails) ====
SERVER_PORT = 7860

def start_cloudflare_tunnel(port):
    import stat
    import subprocess
    import threading
    import urllib.request
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return

    def _watch():
        for line in tunnel.stdout:
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if match:
                print(f"* Cloudflare tunnel URL: {match.group(0)}  (use this if the Gradio link does not load)")
                sys.stdout.flush()
                break
        for _ in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            pass

    threading.Thread(target=_watch, daemon=True).start()

print("\nLaunching Gradio interface...")
sys.stdout.flush()
start_cloudflare_tunnel(SERVER_PORT)
demo.queue()
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, debug=True,
            show_error=True, ssr_mode=False, allowed_paths=[OUTPUT_DIR])

---
## Step 4: Launch! 🚀

Wait for **`✅ Model ready`**, then open the **Gradio public link** or the **Cloudflare link** (`*.trycloudflare.com`) - use the Cloudflare one if the Gradio link does not load.

In [ ]:
!cd /kaggle/working && python -u run_krea_turbo.py 2>&1

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---